In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering
)
import torch


# ---------- Text Summarization ----------
summarizer_model = "facebook/bart-large-cnn"

summarizer_tokenizer = AutoTokenizer.from_pretrained(
    summarizer_model
)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    summarizer_model
)

article = """Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language Models (LLMs)
such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of
natural language tasks including translation, summarization, and question answering. These
models are increasingly being deployed in industry applications ranging from customer support
to software development, transforming how humans interact with machines."""

summary_inputs = summarizer_tokenizer(
    article,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = summarizer_model.generate(
    **summary_inputs,
    max_length=45,
    min_length=20,
    do_sample=False
)

summary = summarizer_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summary:\n", summary)


# ---------- Question Answering ----------
qa_model_name = "distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)

context = article
question = "What are Large Language Models trained on?"

qa_inputs = qa_tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = qa_model(**qa_inputs)

start_index = torch.argmax(outputs.start_logits)
end_index = torch.argmax(outputs.end_logits)

answer_tokens = qa_inputs["input_ids"][0][start_index:end_index + 1]

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

confidence = (
    torch.softmax(outputs.start_logits, dim=1)[0][start_index].item()
)

print("\nQuestion:", question)
print("Answer:", answer)
print("Confidence:", round(confidence, 3))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
 Large Language Models (LLMs) are trained on massive text corpora. They can perform a wide range of natural language tasks including translation, summarization, and question answering.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Question: What are Large Language Models trained on?
Answer: massive text corpora
Confidence: 0.894
